# 00 — NASA API Discovery

**Goal:** Use NASA's open data catalog (CKAN) and Zenodo APIs to locate the AI4Mars dataset metadata and download links.

We will **not** download the full dataset in this notebook — that step is manual to avoid committing large binary files to the repository.

---

## ⚠️ Important — there are two different "AI4Mars" datasets on Zenodo

JPL has published **two separate datasets** that are easy to confuse:

| Dataset | Zenodo record | Contents | Use for this project? |
|---|---|---|---|
| **MSL Curiosity Rover Images with Science and Engineering Classes** | `4033453` | 6,820 images, each with a single **whole-image** class label (19 classes: wheel, sand, arm cover, etc.) in `train/val/test-set*.txt` | ❌ No — this is a scene *classification* dataset, not segmentation |
| **AI4Mars: A Dataset for Terrain-Aware Autonomous Driving on Mars** | `15995036` | ~425K **per-pixel** semantic segmentation masks over ~50K images (soil/bedrock/sand/big-rock), from Curiosity, Perseverance, Opportunity, and Spirit | ✅ Yes — this is the correct dataset for `src/dataset.py` |

If you previously downloaded `msl-labeled-data-set-v2.1.zip`, that is the **classification** dataset (record `4033453`) — it contains no pixel masks, which is why `01_dataset_inspection.ipynb` reports "No pairs found". Re-download using record `15995036` below.

## Why two APIs?

| Portal | Purpose |
|---|---|
| **data.nasa.gov** | NASA's open-data metadata catalog (CKAN). Good for *discovering* datasets, but does not always host the actual files. |
| **zenodo.org** | Open research data repository. The correct AI4Mars segmentation files (~16 GB) are hosted here as a **Zenodo record**. |

The workflow is:
1. Use NASA CKAN to find the dataset name / description.
2. Use Zenodo to get the exact file list and download URLs.
3. Download manually (or with `wget`/`curl`) and extract into `data/raw/`.


In [1]:
import requests
import pandas as pd
from pprint import pprint

# Add the project root to the Python path so we can import from src/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.nasa_catalog import (
    search_nasa_datasets,
    get_nasa_dataset,
    get_zenodo_record,
    list_zenodo_files,
    print_nasa_search_results,
    print_zenodo_files,
)

In [2]:
# Search NASA CKAN for AI4Mars-related datasets
search_response = search_nasa_datasets("AI4Mars Mars terrain autonomous driving", rows=10)

# Pretty-print the titles and dataset names
print_nasa_search_results(search_response)

Found 2 dataset(s):

  [0] title : AI4MARS: A Dataset for Terrain-Aware Autonomous Driving on Mars
       name  : ai4mars-a-dataset-for-terrain-aware-autonomous-driving-on-mars

  [1] title : Human Robotic Systems (HRS): Extreme Terrain Mobility Element
       name  : human-robotic-systems-hrs-extreme-terrain-mobility-element



## Inspect a Specific Dataset

If you found an AI4Mars dataset in the search above, replace `dataset_id` below with the exact name from the `name` field.

In [3]:
# Replace with the actual dataset name from the search results above.
# This may fail if NASA CKAN doesn't have a dedicated AI4Mars entry.
dataset_id = "ai4mars-a-dataset-for-terrain-aware-autonomous-driving-on-mars"

try:
    dataset_info = get_nasa_dataset(dataset_id)
    print(f"Title      : {dataset_info['result'].get('title', 'N/A')}")
    print(f"Description: {dataset_info['result'].get('notes', 'N/A')[:300]}...")
except Exception as e:
    print(f"Could not load dataset '{dataset_id}': {e}")
    print("This is expected if NASA CKAN does not have a direct entry — proceed to Zenodo below.")

Title      : AI4MARS: A Dataset for Terrain-Aware Autonomous Driving on Mars
Description: This dataset was built for training and validating terrain classification models for Mars, which may be useful in future autonomous rover efforts. It consists of ~326K semantic segmentation full image labels on 35K images from Curiosity, Opportunity, and Spirit rovers, collected through crowdsourcin...


## Zenodo Record Metadata

The **correct** AI4Mars terrain-segmentation dataset is published on Zenodo.
The record ID is **15995036** — you can verify it at https://zenodo.org/records/15995036

> Do not use record `4033453` — that is a different, unrelated MSL image-classification dataset (see the warning above).


In [ ]:
# Zenodo record ID for AI4Mars (terrain segmentation dataset — NOT the MSL classification dataset)
# Change this if a newer version has been published.
ZENODO_RECORD_ID = 15995036

record = get_zenodo_record(ZENODO_RECORD_ID)

print(f"Title      : {record.get('metadata', {}).get('title', 'N/A')}")
print(f"Description: {record.get('metadata', {}).get('description', 'N/A')[:400]}...")
print(f"DOI        : {record.get('doi', 'N/A')}")
print(f"Published  : {record.get('metadata', {}).get('publication_date', 'N/A')}")


Title      : MSL Curiosity Rover Images with Science and Engineering Classes
Description: <p>&nbsp;</p>

<p><strong>Please note that the file msl-labeled-data-set-v2.1.zip</strong><strong>&nbsp;below contains the latest images and labels associated with this data set.&nbsp;</strong></p>

<p>&nbsp;</p>

<p><strong>Data Set Description</strong></p>

<p>The data set consists of 6,820 images that were collected by the Mars Science Laboratory (MSL) Curiosity Rover by three instruments: (1) ...
DOI        : 10.5281/zenodo.4033453
Published  : 2020-09-16


## Step 4 — List Zenodo Files

This shows the individual files in the Zenodo record, their sizes, and direct download URLs.

In [5]:
files = list_zenodo_files(record)
print(f"Found {len(files)} file(s) in Zenodo record {ZENODO_RECORD_ID}:\n")
print_zenodo_files(files)

Found 2 file(s) in Zenodo record 4033453:

Filename                                            Size (MB)  Download URL
------------------------------------------------------------------------------------------------------------------------
msl-labeled-data-set.zip                                 54.9  https://zenodo.org/api/records/4033453/files/msl-labeled-data-set.zip/content
msl-labeled-data-set-v2.1.zip                            54.9  https://zenodo.org/api/records/4033453/files/msl-labeled-data-set-v2.1.zip/content


In [6]:
# Display as a tidy DataFrame for easier reading
df = pd.DataFrame([
    {
        "filename": f.get("key"),
        "size_mb": round(f.get("size", 0) / 1_000_000, 1),
        "download_url": f.get("links", {}).get("self", "N/A"),
    }
    for f in files
])
df

,filename,size_mb,download_url
0,msl-labeled-data-set.zip,54.9,https://zenodo.org/api/records/4033453/files/m...
1,msl-labeled-data-set-v2.1.zip,54.9,https://zenodo.org/api/records/4033453/files/m...


## Next Steps

1. Copy the download URL for **`ai4mars-dataset-merged-0.6.zip`** from the table above (this is the merged, ready-to-use version with per-pixel masks; ~16 GB).
2. Download the archive manually (or use `wget`/`curl` in a terminal) into `data/raw/`.
3. Extract the archive: `unzip ai4mars-dataset-merged-0.6.zip -d data/raw/`
4. **Delete or move aside** the old `data/raw/msl-labeled-data-set-v2.1/` folder — it belongs to the wrong (classification) dataset and will confuse the pairing logic in `01_dataset_inspection.ipynb`.
5. Open `01_dataset_inspection.ipynb` to verify the extracted structure.

> **Tip:** The full dataset is large (~16 GB). The `ai4mars-labels-unmerged.zip` (~1.5 GB) file contains only the raw per-annotator labels and is not needed for training — skip it unless you specifically need unmerged annotations.
